# C7-cnn-transfer — Review

Work through this **60-minute** notebook after all four lesson sessions and
the practice sets. Commit answers before opening the final section.

## Formula, API, and state summary

| Register | Contract |
|---|---|
| Convolution shape | $n_{out}=\lfloor(n+2p-K)/s\rfloor+1$, separately for height and width |
| Feature flow | `(N,C,H,W)` → convolution/pool grids → adaptive pool → `Flatten(1)` → `(N,F)` → logits `(N,K)` |
| Receptive field | $r\leftarrow r+(K-1)J$ then $J\leftarrow Js$ |
| CE | `nn.CrossEntropyLoss()(logits, integer_targets)`; no model softmax |
| Training lifecycle | `zero_grad(set_to_none=True) → forward → loss → backward → step` |
| Freezing | set `requires_grad` before optimizer construction; membership is checked by parameter identity |
| Mode | `train()` enables dropout and BN batch-stat/buffer updates; `eval()` disables dropout and makes BN read stored buffers |
| Graph | `eval()` does not disable gradients; `no_grad()` does not change module mode |
| Certificate | finite loss reduction + exact optimizer ownership + expected gradients + allowed movement + frozen immobility + separate buffer audit |

ResNet-50 still follows stem → `layer1..4` → pool → head, bottleneck block
counts `[3,4,6,3]`, and feature channels `256,512,1024,2048`. An interior
bottleneck has convolution count $17m^2$ and BN affine count $12m$.

## Shape / mode / freezing / training audit

For any CNN construction, write six ledgers before interpreting a loss:

1. **Shapes:** every `(N,C,H,W)` and the flattened head width.
2. **Flags:** the exact qualified names with `requires_grad=True`.
3. **Ownership:** optimizer parameter identities equal the intended trainable
   identities—no missing and no extra objects.
4. **Gradients:** after backward, expected parameters have gradients and
   frozen/disconnected parameters do not.
5. **Movement:** compare cloned before/after parameters; frozen ones must be
   bitwise unchanged and at least one allowed parameter must move.
6. **Buffers/mode:** snapshot BatchNorm running state; training may update it,
   evaluation must not. Repeated eval logits should match at named tolerances.

## Checkpoints

1. Trace `(2,3,31,35)` through `Conv2d(3,8,k=5,s=2,p=2)`,
   `MaxPool2d(k=3,s=2,p=1)`, and `AdaptiveAvgPool2d((2,3))`; give the flattened
   width.
2. Which line constructs gradients? Which line mutates parameters? Which
   operation can mutate BatchNorm buffers before either one?
3. Why must freezing occur before optimizer construction, and how do you audit
   ownership robustly?
4. A frozen BatchNorm layer's `running_mean` moves. Is the freeze broken?
5. Validation uses `no_grad()` but repeated logits differ and buffers move.
   Diagnose and repair.
6. Loss falls, all frozen parameters stay fixed, but no trainable parameter
   moves. Name two audits that locate the fault.
7. Why is an exact final-weight assertion weaker engineering than a bundle of
   loss/movement/membership invariants?
8. A model's adaptive pool returns `(N,12,2,2)`. What is the correct input
   width of a linear head, and where must flatten occur?
9. State the difference among a parameter, its `.grad`, an optimizer state
   tensor, and a BatchNorm buffer.
10. After selectively unfreezing `conv2`, why should the optimizer normally be
    rebuilt?

## Exam connections and redo map

- Shape errors: redo `p02`, `p21`, `p23`, `p24`, `p26`.
- Convolution/receptive fields: redo `p01`, `p05`–`p07`, `p13`, `p20`.
- Architecture/counting/truncation: redo `p04`, `p08`–`p09`, `p12`,
  `p14`–`p16`.
- Freezing/transfer: redo `p10`, `p11`, `p16`, `p17`, `p19`, `p27`.
- End-to-end CNN training and mode/buffer audits: redo Session 4 and capstones
  `p10`, `p24`, `p26`, `p27`.

## Answers

<details><summary><b>Open after committing all ten answers</b></summary>

1. Conv: `(2,8,16,18)`; pool: `(2,8,8,9)`; adaptive pool:
   `(2,8,2,3)`; flattened width `8·2·3=48`.
2. `backward()` constructs/accumulates gradients; `step()` mutates owned
   parameters; a training-mode BatchNorm forward can mutate running buffers.
3. Optimizers capture objects at construction. Compare the set of `id(p)` in
   all parameter groups with the `id(p)` set for intended trainable parameters.
4. Not necessarily. Freezing controls affine parameter gradients; training
   mode independently updates running buffers.
5. The model remained in training mode. Call `eval()` and use `no_grad()`, then
   certify repeated logits and unchanged buffers.
6. Check optimizer ownership and gradients; the parameter may be omitted from
   the optimizer or disconnected/have no effective gradient.
7. Valid deterministic implementations can land at different exact weights;
   behavioral and lifecycle invariants test the intended contract directly.
8. `12·2·2=48`; use `Flatten(1)` after pooling and before the linear head.
9. A parameter is model state optimized by gradients; `.grad` is accumulated
   derivative state; optimizer state belongs to the update rule; a BN buffer is
   persistent model state updated by module forward logic.
10. Existing optimizers do not discover newly trainable objects; rebuilding
    makes membership exactly match the new intended set.

</details>
